# 🎓 UC4 - Machine Learning e Data Mining (Analista de Dados)
## 🧪 Aula 01 (01-07) - Revisão Prática de Pandas: Limpeza e Transformação de Dados

Este notebook foi preparado para guiar a nossa prática de revisão. Vamos trabalhar com um dataset dinâmico gerado na hora a partir da sua matrícula, garantindo dados exclusivos para cada aluno.

### 🎯 Nosso Roteiro:
1. **Gerar nossa base de treino** (`vendas_treino.csv`) usando sua matrícula do Senac.
2. **Limpar os dados** (tratar valores ausentes, duplicados e outliers via IQR).
3. **Normalizar** variáveis numéricas (escala Min-Max).
4. **Transformar** dados categóricos e criar classificações com `np.where` e `lambda`.
5. **Desafio Final**: Responder perguntas de faturamento e ticket médio da *sua* base individualizada.

---
### ⚙️ Passo 0: Inicialização do Dataset Dinâmico
Rode a célula abaixo, digite sua matrícula (ou seu primeiro nome) no campo de texto e pressione `Enter` para gerar seu arquivo de vendas exclusivo.

In [ ]:
import hashlib
import pandas as pd
import numpy as np
import random

# 1. Obter semente exclusiva a partir da matrícula
matricula = input("Digite sua matrícula (ou primeiro nome): ").strip()
if not matricula:
    matricula = "Senac"
seed_usuario = int(hashlib.sha256(matricula.encode('utf-8')).hexdigest(), 16) % (10**8)

np.random.seed(seed_usuario)
random.seed(seed_usuario)

# 2. Gerar 150 registros com ruídos e anomalias controladas
n_registros = 150
precos_base = {
    "Smartphone": 1500, "Notebook": 3200, "Fone de Ouvido": 250, "Tablet": 1000, "Smartwatch": 750
}
produtos = list(precos_base.keys())

registros = []
for i in range(1, n_registros + 1):
    prod = random.choice(produtos)
    qtd = random.randint(1, 5)
    # 10% de chance de quantidade nula
    if random.random() < 0.10:
        qtd = np.nan
        
    valor_unit = precos_base[prod] * np.random.uniform(0.9, 1.1)
    # 5% de chance de preço unitário nulo
    if random.random() < 0.05:
        valor_unit = np.nan
        
    desconto = round(random.choice([0, 0, 0.05, 0.10, 0.15]), 2)
    # 5% de chance de desconto nulo
    if random.random() < 0.05:
        desconto = np.nan
        
    categoria = random.choice(["VIP", "vip", "Vip", "Regular", "regular", "Bronze", "bronze", np.nan])
    fidelidade = random.randint(50, 950)
    
    registros.append({
        "id_venda": i,
        "produto": prod,
        "quantidade": qtd,
        "valor_unitario": round(valor_unit, 2) if not np.isnan(valor_unit) else np.nan,
        "desconto": desconto,
        "categoria_cliente": categoria,
        "pontuacao_fidelidade": fidelidade
    })

df_init = pd.DataFrame(registros)

# Injetar 1 outlier extremo de digitação
outlier_idx = random.randint(0, n_registros - 1)
df_init.loc[outlier_idx, "valor_unitario"] = 145000.0

# Injetar 5 duplicatas completas
duplicatas = df_init.sample(n=5, random_state=seed_usuario)
df_init = pd.concat([df_init, duplicatas], ignore_index=True)
df_init = df_init.sample(frac=1, random_state=seed_usuario).reset_index(drop=True)

df_init.to_csv("vendas_treino.csv", index=False)
print(f"\n🎉 Base 'vendas_treino.csv' gerada com sucesso para a semente/matrícula: {matricula}!")
print(f"Total de registros gerados: {len(df_init)}")

---
### 📖 Passo 1: Inspeção Geral dos Dados
Vamos carregar o arquivo gerado e analisar o estado da tabela.

In [ ]:
df = pd.read_csv('vendas_treino.csv')
print("Formato da tabela (linhas, colunas):", df.shape)
df.head(10)

---
### 🧹 Passo 2: Tratamento de Nulos e Duplicados
Nesta etapa, localizaremos e preencheremos os valores nulos e removeremos registros duplicados.

In [ ]:
# 1. Verificar nulos por coluna
print("Valores ausentes por coluna:")
print(df.isna().sum())

# 2. Tratar quantidade nula: Preencher com a mediana da coluna
mediana_qtd = df['quantidade'].median()
df['quantidade'] = df['quantidade'].fillna(mediana_qtd)

# 3. Tratar valor_unitario nulo: Preencher com a mediana por produto (mais inteligente)
df['valor_unitario'] = df.groupby('produto')['valor_unitario'].transform(lambda x: x.fillna(x.median()))

# 4. Tratar desconto nulo: Assumir que nulo equivale a 0% de desconto (0.0)
df['desconto'] = df['desconto'].fillna(0.0)

# 5. Remover registros duplicados
linhas_antes = len(df)
df = df.drop_duplicates()
linhas_depois = len(df)
print(f"\nRegistros duplicados removidos: {linhas_antes - linhas_depois}")
print("Valores nulos após tratamento:")
print(df.isna().sum())

---
### 📊 Passo 3: Remoção de Outliers via IQR
Identificar se há preços com valores impossíveis (como o erro de digitação de 145.000) e filtrá-los estatisticamente.

In [ ]:
# Calcular os limites estatísticos (IQR) do valor unitário
Q1 = df['valor_unitario'].quantile(0.25)
Q3 = df['valor_unitario'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Limite Inferior: {limite_inferior} | Limite Superior: {limite_superior}")

# Filtrar o dataset mantendo apenas valores dentro dos limites
df_filtrado = df[(df['valor_unitario'] >= limite_inferior) & (df['valor_unitario'] <= limite_superior)].copy()
print(f"Linhas antes do IQR: {len(df)} | Linhas após o IQR: {len(df_filtrado)}")

---
### 📏 Passo 4: Normalização de Escala (Min-Max)
Mapear a pontuação de fidelidade do cliente para que fique restrita ao intervalo [0, 1].

In [ ]:
min_fid = df_filtrado['pontuacao_fidelidade'].min()
max_fid = df_filtrado['pontuacao_fidelidade'].max()

df_filtrado['fidelidade_normalizada'] = (df_filtrado['pontuacao_fidelidade'] - min_fid) / (max_fid - min_fid)
df_filtrado[['pontuacao_fidelidade', 'fidelidade_normalizada']].head(10)

---
### 🔄 Passo 5: Padronização de Categorias e Criação de Indicadores
Corrigir typos na coluna de categoria e calcular a receita total.

In [ ]:
# 1. Verificar categorias do cliente antes
print("Categorias únicas antes:", df_filtrado['categoria_cliente'].unique())

# 2. Padronizar textos (tudo em maiúscula, sem espaços em branco e preencher nulos com 'REGULAR')
df_filtrado['categoria_cliente'] = df_filtrado['categoria_cliente'].str.upper().str.strip()
df_filtrado['categoria_cliente'] = df_filtrado['categoria_cliente'].fillna('REGULAR')
print("Categorias únicas depois:", df_filtrado['categoria_cliente'].unique())

# 3. Criar cálculo de Valor Total da Venda (Quantidade * Valor_Unitario * (1 - Desconto))
df_filtrado['valor_total'] = df_filtrado['quantidade'] * df_filtrado['valor_unitario'] * (1 - df_filtrado['desconto'])

# 4. Usar np.where para criar uma classificação da venda
df_filtrado['tipo_receita'] = np.where(df_filtrado['valor_total'] > 5000.0, 'Faturamento Alto', 'Faturamento Padrão')

df_filtrado.head(10)

---
## 🎯 DESAFIO PRÁTICO INDIVIDUAL (Anti-IA)
Agora que o pipeline está pronto, execute os cálculos abaixo sobre a **sua** tabela resultante (`df_filtrado`) e registre as respostas. Como os dados foram gerados a partir da sua matrícula, os valores serão únicos!

### Perguntas a Responder:
1. **Qual o Faturamento Total acumulado na sua base limpa?** (Soma da coluna `valor_total`)
2. **Quantos registros válidos restaram na sua tabela após a remoção de duplicados e de outliers?** (Quantidade de linhas total de `df_filtrado`)
3. **Qual foi o maior desconto concedido na sua base?** (Máximo da coluna `desconto`)
4. **Qual o faturamento médio dos clientes categorizados como VIP?**

*(Escreva os códigos para obter as respostas nas células abaixo)*

In [ ]:
# Código para pergunta 1 (Faturamento Total)


In [ ]:
# Código para pergunta 2 (Total de Linhas Válidas)


In [ ]:
# Código para pergunta 3 (Maior Desconto)


In [ ]:
# Código para pergunta 4 (Faturamento Médio VIP)
